# 03 Stacking 학습·검증·보고서

OLS·Ridge·XGBoost 예측을 Ridge 메타 모델로 결합합니다. 원본 MAT 없이 동봉 특징 CSV로 실행할 수 있습니다. 학습→평가→보고서 셀을 순서대로 실행하며 매 실행은 새 결과 폴더에 저장합니다. MAPE는 낮을수록 더 정확하며 Gap은 Valid−Train, Test−Valid, Test−9.1입니다.


In [1]:
from pathlib import Path
from datetime import datetime
import sys, json
import pandas as pd
from IPython.display import display, Markdown, Image
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/pipeline.py').is_file())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.pipeline import load_config
from src.runtime import run_pipeline
CONFIG = load_config(ROOT / 'config/model.json')
RUN_DIR = ROOT / 'results/runs' / ('notebook_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
MODEL_DIR = RUN_DIR / 'model'
print('프로젝트:', ROOT)
print('새 실험 결과:', RUN_DIR)


프로젝트: .
새 실험 결과: ./results/runs/notebook_20261002_145315_690936


## 1 특징 CSV와 고정 그룹 분할

원본이 없어도 동봉 특징 CSV로 이 단계부터 실행할 수 있습니다. B1은 같은 충전 프로토콜이 학습/holdout 양쪽에 들어가지 않게 분리합니다.


In [2]:
print('학습 모델:', CONFIG['model_type'])
print('필요 패키지는 프로젝트 requirements.txt로 설치합니다.')


학습 모델: stacking
필요 패키지는 프로젝트 requirements.txt로 설치합니다.


In [3]:
from src.pipeline import read_feature_csv
from src.splits import make_splits
frame = read_feature_csv(ROOT / 'data/processed/cell_features.csv', CONFIG)
manifest = make_splits(frame, CONFIG)
display(manifest.groupby(['batch', 'role']).size().rename('n'))


batch  role   
B1     train      35
       valid      11
B2     test_b2    39
B3     test_b3    44
Name: n, dtype: int64

## 2 B1 내부 학습과 그룹 CV

충전 프로토콜로 분리한5fold를3회 반복합니다. 각 학습 부분 안에서 후보 상관순위로2개를 선택하고 XGBoost12설정을 내부3fold로 비교합니다. 메타 Ridge는 그룹 OOF 기저 예측으로 학습합니다. 메타 예측을 만들 때도 해당 셀의 정답으로 특징·조정값을 고르지 않습니다.

최종 B1 학습35개에서는 ΔQ 로그분산·RMS·ΔQ 5% 분위수·평균을 선택했습니다. 목표는log10 수명이며 점수는 원래 사이클 단위로 역변환해 계산합니다.


In [4]:
RUN_TRAINING = True
if RUN_TRAINING:
    run_pipeline(ROOT, ['train', '--run-dir', RUN_DIR])
    selected = json.loads((MODEL_DIR / 'metadata.json').read_text())
    print('선택 모델:', selected['model'])
    print('최종 입력:', selected['selected_features'])
    print('최종 설정:', selected['parameters'])
    display(pd.read_csv(RUN_DIR / 'cv_repeat_results.csv'))
else:
    print('새 학습을 실행하지 않았습니다.')


Stacking 그룹 CV 1/3: MAPE 6.497%


Stacking 그룹 CV 2/3: MAPE 6.727%


Stacking 그룹 CV 3/3: MAPE 6.711%


학습 완료: Stacking (OLS + Ridge + XGBoost → Ridge), ['dq100_10_log_variance', 'cycle10_rms_c_timeweighted', 'dq100_10_q05_ah', 'dq100_10_mean_ah']; holdout/B2 평가 전
선택 모델: Stacking (OLS + Ridge + XGBoost → Ridge)
최종 입력: ['dq100_10_log_variance', 'cycle10_rms_c_timeweighted', 'dq100_10_q05_ah', 'dq100_10_mean_ah']
최종 설정: {'xgb': {'reg_lambda': 10.0, 'n_estimators': 250, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.03}, 'meta_alpha': 1.0}


,repeat,seed,MAPE_percent,MAE_cycles,RMSE_cycles
0,1,42,6.496736,52.970175,71.099882
1,2,7,6.726999,53.899197,70.146263
2,3,2026,6.711153,56.394350,73.246385


## 3 동일 저장 모델로 검증·평가

B1 holdout11개/B2 39개/B3 44개를 평가합니다. 평가 단계에서는 다시 학습하거나 특징·조정값을 선택하지 않습니다. 기존 결과를 덮어쓰지 않습니다.


In [5]:
RUN_EVALUATION = True
INCLUDE_B3 = True
if RUN_EVALUATION:
    arguments = ['evaluate', '--run-dir', RUN_DIR]
    if INCLUDE_B3: arguments.append('--include-b3')
    run_pipeline(ROOT, arguments)
    display(pd.read_csv(RUN_DIR / 'model_performance.csv')[['구분', 'MAPE (%)', '비고']])
else:
    print('저장 모델 평가를 실행하지 않았습니다.')


                         구분   MAPE (%)                                  비고
         Train (Batch 1 CV)   6.644963        B1 그룹 OOF CV; 낮을수록 예측이 더 정확함
   Valid (Batch 1 Hold-out)  11.384574                B1 holdout; 동일 저장 모델
             Test (Batch 2)  27.716713                         필수 최종 배치 평가
          Gap (Train-Valid)   4.739611  Valid−Train (%p); (+) 과적합/집단 차이 검토
           Gap (Valid-Test)  16.332140 Test−Valid (%p); (+) 배치 간 일반화 저하 검토
          Gap (Target-Test)  18.616713  Test−9.1 (%p); 논문 Target과 다른 실험 구성
             Test (Batch 3)  12.779998                         동일 모델 추가 평가
        Gap (Batch2-Batch3) -14.936716          B3−B2 (%p); (+) B3에서 오차 증가
Gap (Target-Test) [Batch 3]   3.679998               B3−9.1 (%p); 추가 참고 비교
평가 결과 저장: ./results/runs/notebook_20261002_145315_690936; report 명령으로 도표와 보고서를 만드세요.


,구분,MAPE (%),비고
0,Train (Batch 1 CV),6.644963,B1 그룹 OOF CV; 낮을수록 예측이 더 정확함
1,Valid (Batch 1 Hold-out),11.384574,B1 holdout; 동일 저장 모델
2,Test (Batch 2),27.716713,필수 최종 배치 평가
3,Gap (Train-Valid),4.739611,Valid−Train (%p); (+) 과적합/집단 차이 검토
4,Gap (Valid-Test),16.332140,Test−Valid (%p); (+) 배치 간 일반화 저하 검토
5,Gap (Target-Test),18.616713,Test−9.1 (%p); 논문 Target과 다른 실험 구성
6,Test (Batch 3),12.779998,동일 모델 추가 평가
7,Gap (Batch2-Batch3),-14.936716,B3−B2 (%p); (+) B3에서 오차 증가
8,Gap (Target-Test) [Batch 3],3.679998,B3−9.1 (%p); 추가 참고 비교


## 4 성능 보고서와 오류 분석

주 지표 MAPE는 평균 상대 오차, MAE는 평균 cycle 오차, RMSE는 큰 오차를 보여줍니다. 예를 들어 Train 8%/Valid 12%이면 Gap=12−8=+4%p입니다. 평가셋별 APE 상위 3개 오류 사례와 실제/예측·상대 오차·입력 분포 차이 그림을 만듭니다.


In [6]:
RUN_REPORT = True
if RUN_REPORT:
    run_pipeline(ROOT, ['report', '--run-dir', RUN_DIR])
    display(Markdown((RUN_DIR / 'performance_report.md').read_text()))
else:
    print('보고서를 생성하지 않았습니다.')


성능 보고서: ./results/runs/notebook_20261002_145315_690936/performance_report.md


# 성능 결과

MAPE가 낮을수록 수명 예측이 더 정확합니다. Gap의 단위는%p이며 양수는 뒤 평가의 오차 증가입니다.

| 구분 | MAPE (%) | 비고 |
| --- | --- | --- |
| Train (Batch 1 CV) | 6.645 | B1 그룹 OOF CV; 낮을수록 예측이 더 정확함 |
| Valid (Batch 1 Hold-out) | 11.385 | B1 holdout; 동일 저장 모델 |
| Test (Batch 2) | 27.717 | 필수 최종 배치 평가 |
| Gap (Train-Valid) | 4.740 | Valid−Train (%p); (+) 과적합/집단 차이 검토 |
| Gap (Valid-Test) | 16.332 | Test−Valid (%p); (+) 배치 간 일반화 저하 검토 |
| Gap (Target-Test) | 18.617 | Test−9.1 (%p); 논문 Target과 다른 실험 구성 |
| Test (Batch 3) | 12.780 | 동일 모델 추가 평가 |
| Gap (Batch2-Batch3) | -14.937 | B3−B2 (%p); (+) B3에서 오차 증가 |
| Gap (Target-Test) [Batch 3] | 3.680 | B3−9.1 (%p); 추가 참고 비교 |

## 보조 지표와 중앙 수명 기준 모델

| scope | model | MAPE_percent | MAE_cycles | RMSE_cycles | n | n_repeats |
| --- | --- | --- | --- | --- | --- | --- |
| train_cv | Stacking (OLS + Ridge + XGBoost → Ridge) | 6.645 | 54.421 | 71.498 | 35 | 3.000 |
| valid | Stacking (OLS + Ridge + XGBoost → Ridge) | 11.385 | 107.739 | 134.866 | 11 | — |
| test_b2 | Stacking (OLS + Ridge + XGBoost → Ridge) | 27.717 | 138.527 | 154.494 | 39 | — |
| test_b3 | Stacking (OLS + Ridge + XGBoost → Ridge) | 12.780 | 159.621 | 249.636 | 44 | — |
| train_cv | Training median | 19.887 | 150.700 | 185.326 | 35 | 3.000 |
| valid | Training median | 16.488 | 154.818 | 192.923 | 11 | — |
| test_b2 | Training median | 72.400 | 341.051 | 364.606 | 39 | — |
| test_b3 | Training median | 20.232 | 251.341 | 370.602 | 44 | — |

MAE는 평균 사이클 오차, RMSE는 큰 오차의 영향을 보여줍니다. 중앙 수명 기준값은 CV에서 각 fold 학습 정답의 중앙값, Valid/B2/B3에서 학습35개 정답의 중앙값입니다.

## 해석 범위

Train은 B1 내부 그룹 OOF CV 점수이며 재대입 성능이 아닙니다. Stacking은 3반복×5fold의 반복별 점수 평균이며 독립 학습 셀은 35개입니다. B2/B3는 EDA와 모델 비교에서 확인했으므로 완전히 미관측한 외부 자료는 아닙니다. 논문9.1%는 과제 참고 Target이며, 이번 코호트·분할·라벨 규칙으로 정확한 논문 재현을 주장하지 않습니다.

![실제값과 예측값](figures/01_actual_vs_predicted.png)

![부호 있는 상대 오차](figures/02_signed_error.png)

![입력 특징 분포 차이](figures/03_feature_distribution_shift.png)

[큰 오차 사례](error_analysis.md)


## 해석 범위

Train은 반복별 OOF 점수의 평균입니다. 독립 학습 셀은35개이며105개가 아닙니다. B2/B3는 EDA와 모델 비교에 이미 사용했고, 배치별 수명 라벨의 종료 규칙도 다릅니다. 논문9.1%는 과제 Target으로 비교하며 동일 실험 재현을 주장하지 않습니다. ESS 적용에는 calendar aging·온도/SOC·팩 차이·불확실성과 새로운 외부 자료 검증이 필요합니다.
